Steps to be performed

1. import the library

2. load the documents

3. split the documents in the chunk

4. create the embeddings

5. store the embedding into chroma

6. create the retriever

7. initialize the language model

8. build the retrieved base QA chain

9. Ask a question

10. observe and explain the output

In [ ]:
!pip install langchain==0.1.16 langchain-community langchain-openai

In [ ]:
pip install pypdf

In [ ]:
pip install chromadb

In [ ]:
#import the library
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.chat_models import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Set up the API key from environment variable
# Create a .env file with: OPENAI_API_KEY=sk-proj-[your-key-here]
api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    os.environ['OPENAI_API_KEY'] = api_key
else:
    print('Please set OPENAI_API_KEY in your .env file')

#load the pdf file
pdf_path='/content/bhagavad-gita-in-english-source-file-2.pdf'
loader=PyPDFLoader(pdf_path)
documents=loader.load()

In [ ]:
#split the text into chunks
splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)
chunks=splitter.split_documents(documents)

In [ ]:
#create the embedding and store in the chroma vector store
embeddings=OpenAIEmbeddings()
vector_store=Chroma.from_documents(chunks,embedding=embeddings)

In [ ]:
#create retrieval based QA chain
retriever=vector_store.as_retriever()
llm=ChatOpenAI(model_name='gpt-4o-mini',temperature=0)
qa_chain=RetrievalQA.from_chain_type(
    llm,
    retriever=retriever,
    return_source_documents=True
)

In [ ]:
#ask the question
query='who is Arjun?'
result=qa_chain(query)
#print the answer
print(result)

In [ ]:

#print the source
for doc in result['source_documents']:
  print(f'page content:{doc.page_content[:200]}')